In [46]:
import pandas as pd
import json

In [6]:
lang = pd.read_csv("data/iso_639-1.csv")
reports = pd.read_csv("rsna_report_translations.csv")

In [ ]:
lang.columns

Index(['family', 'name', 'nativeName', '639-1', '639-2', '639-2/B'], dtype='str')

In [9]:
lang.head()

,family,name,nativeName,639-1,639-2,639-2/B
0,Northwest Caucasian,Abkhaz,"аҧсуа бызшәа, аҧсшәа",ab,abk,NaN
1,Afro-Asiatic,Afar,Afaraf,aa,aar,NaN
2,Indo-European,Afrikaans,Afrikaans,af,afr,NaN
3,Niger–Congo,Akan,Akan,ak,aka,NaN
4,Indo-European,Albanian,Shqip,sq,sqi,alb


In [11]:
languages = lang[["name", "639-1"]].rename({"639-1":"language_code"}, axis = 1)

In [12]:
reports.columns

Index(['Unnamed: 0', 'StudyInstanceUID', 'Report',
       'English_Translation_qwen2.5:3b', 'English_Translation_llama3.2:3b',
       'English_Translation_gemma2:2b', 'name'],
      dtype='str')

In [20]:
report_data = (reports
               .drop('Unnamed: 0', axis=1)
               .melt(id_vars=['StudyInstanceUID', 'Report', 'name'], 
                    value_vars=['English_Translation_qwen2.5:3b', 'English_Translation_llama3.2:3b','English_Translation_gemma2:2b'],
                    var_name="model",
                    value_name="choice_text"))

In [21]:
report_data.head()

,StudyInstanceUID,Report,name,model,choice_text
0,1.2.826.0.1.3680043.8.498.10004873229099053869...,Técnica: RMN de la rodilla. Resultados: Rotura...,Spanish,English_Translation_qwen2.5:3b,MRI of the knee technique. Results: Internal m...
1,1.2.826.0.1.3680043.8.498.10004945927472656027...,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,German,English_Translation_qwen2.5:3b,[DATE]: * Right Knee MRI 15ch AA Clinical Info...
2,1.2.826.0.1.3680043.8.498.10009278692606631573...,Hallazgos:\nNo hay alteraciones en significati...,Spanish,English_Translation_qwen2.5:3b,Findings:\nNo significant abnormalities in the...
3,1.2.826.0.1.3680043.8.498.10009639203170750274...,"In the medial compartment, the meniscus is no...",English,English_Translation_qwen2.5:3b,"In the medial compartment, the meniscus is int..."
4,1.2.826.0.1.3680043.8.498.10013663742400736029...,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,French,English_Translation_qwen2.5:3b,Conclusions :\n\nFractures :\nNone.\n\nAlignme...


In [22]:
report_data.columns

Index(['StudyInstanceUID', 'Report', 'name', 'model', 'choice_text'], dtype='str')

In [23]:
report_data.name.value_counts()

name
English           5184
Spanish           2055
Turkish           1638
Greek (modern)     963
German             903
Bulgarian          657
Dutch              342
Croatian           309
French             243
Bosnian             69
Serbian             27
Portuguese          12
Slovene              9
Polish               3
Russian              3
Name: count, dtype: int64

In [25]:
report_data.count()

StudyInstanceUID    13221
Report              13221
name                12417
model               13221
choice_text         13221
dtype: int64

In [24]:
report_data.name.describe()

count       12417
unique         15
top       English
freq         5184
Name: name, dtype: object

In [47]:
t_data = (report_data.assign(other_data = [
        json.dumps({'id': uid, 'model': model})
        for uid, model in zip(
            report_data['StudyInstanceUID'], 
            report_data['model']
        )
    ])
.drop(['StudyInstanceUID', 'model'], axis = 1)
.rename({"Report":"question", "name": 'language'}, axis=1))

In [48]:
languages.to_csv("data/languages_load.csv", index=False)

In [49]:
t_data

,question,language,choice_text,other_data
0,Técnica: RMN de la rodilla. Resultados: Rotura...,Spanish,MRI of the knee technique. Results: Internal m...,"{""id"": ""1.2.826.0.1.3680043.8.498.100048732290..."
1,[DATE]: * MR Knie Rechts 15ch AA Klinische Inl...,German,[DATE]: * Right Knee MRI 15ch AA Clinical Info...,"{""id"": ""1.2.826.0.1.3680043.8.498.100049459274..."
2,Hallazgos:\nNo hay alteraciones en significati...,Spanish,Findings:\nNo significant abnormalities in the...,"{""id"": ""1.2.826.0.1.3680043.8.498.100092786926..."
3,"In the medial compartment, the meniscus is no...",English,"In the medial compartment, the meniscus is int...","{""id"": ""1.2.826.0.1.3680043.8.498.100096392031..."
4,CONSTATATIONS :\n\nFractures :\nAucune.\n\nAli...,French,Conclusions :\n\nFractures :\nNone.\n\nAlignme...,"{""id"": ""1.2.826.0.1.3680043.8.498.100136637424..."
...,...,...,...,...
13216,Exam Type: MRI KNEE RIGHT WO CONTRAST\nExam Da...,English,MRI KNEE RIGHT WITHOUT CONTRAST\n\nDate and Ti...,"{""id"": ""1.2.826.0.1.3680043.8.498.998800212806..."
13217,[DATE]: *MR Knie Rechts 15ch AA Klinische Inli...,German,Right knee MRI 15th week of AA clinical inform...,"{""id"": ""1.2.826.0.1.3680043.8.498.998816203365..."
13218,"SAĞ DİZ MRG. Tetkik protokolü: Çok düzlemli, ç...",Turkish,"Meniscus MRI: Protocol: Multiplanar, multisequ...","{""id"": ""1.2.826.0.1.3680043.8.498.999266249682..."
13219,"MRI of Knee with \n-Locator, SG PD FatSat, SG ...",English,"MRI of Knee with \n-Locator, SG PD FatSat, SG ...","{""id"": ""1.2.826.0.1.3680043.8.498.999391006572..."


In [50]:
t_data.to_csv("data/reports_load.csv", index=False)

In [45]:
COPY polls_language(language_code, name)
FROM '/tmp/languages_load.csv'
WITH (
    FORMAT csv,
    HEADER true,
    DELIMITER ','
);

CREATE TABLE staging_questions (
    question text,
    language text,
    choice_text text,
    other_data jsonb
);

\copy staging_questions (question, language_id, choice_text, other_data) FROM '/tmp/reports_load.csv' WITH (FORMAT csv, HEADER true);


SyntaxError: invalid syntax (1463473191.py, line 1)

In [ ]:
WITH unique_questions AS (
    -- Step A: Deduplicate questions and look up Language ID by name
    SELECT DISTINCT 
        stg.question, 
        l.id AS language_id
    FROM staging_questions stg
    JOIN polls_language l ON l.name = stg.language
),
inserted_questions AS (
    -- Step B: Insert into Question and capture generated IDs
    INSERT INTO polls_question (
        question, 
        language_id, 
        published_date, 
        active
    )
    SELECT 
        question, 
        language_id, 
        NOW(), 
        TRUE
    FROM unique_questions
    RETURNING id, question, language_id
)
-- Step C: Insert into Choice using generated Question IDs and direct jsonb data
INSERT INTO polls_choice (
    question_id, 
    choice_text, 
    other_data
)
SELECT 
    iq.id,
    stg.choice_text,
    stg.other_data
FROM staging_questions stg
JOIN polls_language l ON l.name = stg.language
JOIN inserted_questions iq 
  ON stg.question = iq.question 
 AND l.id = iq.language_id;